# Laboratorio 4 - Machine Learning
### Sergio David Ferreira - Juan David Gutierrez


## 1. Carga de datos y entendimiento del problema

**Objetivo de negocio:** encontrar grupos de pasajeros con caracteristicas y percepciones de servicio similares para apoyar decisiones de segmentacion en VuelaAlpes.

**Plan del notebook**
1. Explorar calidad y estructura de los datos.
2. Proponer y aplicar una limpieza justificada.
3. Construir tres modelos de agrupacion con pipelines y busqueda de hiperparametros.
4. Comparar cuantitativamente los modelos.
5. Interpretar cualitativamente los grupos.
6. Responder las preguntas de analisis de resultados.

In [ ]:
import os
import warnings
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, KMeans, HDBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

os.environ["LOKY_MAX_CPU_COUNT"] = "4"
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")

RANDOM_STATE = 42
SEARCH_SAMPLE_SIZE = 3000

In [ ]:
raw_data = pd.read_csv("data/Datos_VuelaAlpes.csv")
data_dictionary = pd.read_excel("data/Diccionario VuelaAlpes.xlsx")

display(Markdown("### Vista general del dataset"))
display(raw_data.head())

display(Markdown("### Diccionario de datos"))
display(data_dictionary)

print(f"Filas y columnas: {raw_data.shape}")

## 2. Exploracion de los datos

En esta etapa buscamos problemas de calidad, tipos de variables, posibles valores atipicos y patrones iniciales utiles para el agrupamiento.

In [ ]:
quality_report = pd.DataFrame(
    {
        "dtype": raw_data.dtypes.astype(str),
        "missing": raw_data.isna().sum(),
        "missing_pct": (raw_data.isna().mean() * 100).round(2),
        "unique_values": raw_data.nunique(),
    }
).sort_values(["missing", "unique_values"], ascending=[False, False])

numeric_summary = raw_data.describe().T
duplicate_rows = raw_data.duplicated().sum()

display(Markdown("### Reporte de calidad"))
display(quality_report)

display(Markdown("### Resumen estadistico de variables numericas"))
display(numeric_summary)

print(f"Registros duplicados exactos: {duplicate_rows}")

In [ ]:
vif_data = raw_data.select_dtypes(include=[np.number]).dropna()
vif_data['intercept'] = 1
vif_values = [variance_inflation_factor(vif_data.values, i) for i in range(len(vif_data.columns))]
print("VIF para variables numéricas:")
for col, vif in zip(vif_data.columns, vif_values):
    print(f"{col}: {vif:.2f}")

In [ ]:
fig = plt.figure(figsize=(8, 4))
sns.scatterplot(data=raw_data, x='Arrival Delay in Minutes', y='Departure Delay in Minutes')
ax = fig.axes[0]
max_val = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='y=x')
ax.legend()
plt.show()

In [ ]:
no_nulls = raw_data.dropna(subset=['Arrival Delay in Minutes', 'Departure Delay in Minutes'])
correlation = no_nulls['Arrival Delay in Minutes'].corr(no_nulls['Departure Delay in Minutes'])
print(f"Correlación entre 'Arrival Delay in Minutes' y 'Departure Delay in Minutes': {correlation:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(raw_data["Age"], bins=20, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Distribucion de edades")

sns.histplot(raw_data["Flight Distance"], bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Distribucion de distancia de vuelo")

sns.boxplot(data=raw_data[["Departure Delay in Minutes", "Arrival Delay in Minutes"]], ax=axes[1, 0])
axes[1, 0].set_title("Retrasos: presencia de valores extremos")

class_order = raw_data["Class"].value_counts().index
sns.countplot(data=raw_data, x="Class", order=class_order, ax=axes[1, 1])
axes[1, 1].set_title("Distribucion por clase")

plt.tight_layout()
plt.show()

### Hallazgos de la exploracion

- La base tiene **10.000 registros** y mezcla variables numericas, categoricas y escalas ordinales de 0 a 5.
- El unico faltante visible esta en `Arrival Delay in Minutes` con **25 valores nulos**; por su naturaleza numerica, se puede imputar con la mediana.
- `id` es un identificador y no aporta informacion util para agrupamiento, por lo que debe excluirse del modelado.
- Los retrasos presentan cola larga y valores extremos altos. No se eliminan porque pueden representar perfiles reales de operacion y experiencia.
- Las variables categoricas (`Gender`, `Customer Type`, `Type of Travel`, `Class`) deben codificarse para ser utilizadas por los algoritmos.
- Como las escalas y unidades son distintas, es necesario **estandarizar** antes de ajustar los modelos.

## 3. Propuesta de limpieza y preparacion

Decisiones de preparacion:

- Eliminar la columna `Arrival Delay in Minutes` por su alta correlación con `Departure Delay in Minutes` y sus valores faltantes.
- Eliminar `id`.
- Estandarizar variables numericas con `StandardScaler`.
- Codificar variables categoricas con `OneHotEncoder`.
- Integrar estas transformaciones en un `Pipeline` para que el flujo sea reproducible y consistente entre modelos.

In [ ]:
analysis_data = raw_data.copy()
analysis_data = analysis_data.drop(columns=["id", "Arrival Delay in Minutes"])

categorical_features = analysis_data.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_features = analysis_data.select_dtypes(exclude=["object", "string"]).columns.tolist()

numeric_transformer = Pipeline(
    [
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

search_data = analysis_data.sample(
    n=min(SEARCH_SAMPLE_SIZE, len(analysis_data)),
    random_state=RANDOM_STATE,
)

print("Variables numericas:", numeric_features)
print("Variables categoricas:", categorical_features)
print(f"Tamanio de muestra para la busqueda: {search_data.shape}")

## 4. Modelamiento

Se comparan tres enfoques de agrupacion:

- `KMeans`
- `AgglomerativeClustering`
- `HBDScan`

Todos se integran con el mismo preprocesamiento para asegurar una comparacion justa.

In [ ]:
algorithm_spaces = {
    "KMeans": {
        "estimator": KMeans(random_state=RANDOM_STATE),
        "params": {
            "n_clusters": [2, 3, 4, 5, 6],
            "n_init": [10, 20],
        },
    },
    "AgglomerativeClustering": {
        "estimator": AgglomerativeClustering(),
        "params": {
            "n_clusters": [2, 3, 4, 5, 6],
            "linkage": ["ward", "average", "complete"],
        },
    },
    "HDBScan": {
        "estimator": HDBSCAN(),
        "params": {
            "min_cluster_size": [5, 10, 15],
            "cluster_selection_epsilon": [0.5, 1.0, 1.5, 2.0],
            "min_samples": [3, 5, 10],
        },
    },
}


def parameter_grid(param_dict):
    keys = list(param_dict.keys())
    for values in product(*param_dict.values()):
        yield dict(zip(keys, values))


def build_pipeline(estimator):
    return Pipeline(
        [
            ("preprocessor", clone(preprocessor)),
            ("clusterer", estimator),
        ]
    )


def get_cluster_labels(fitted_pipeline, X):
    transformed = fitted_pipeline.named_steps["preprocessor"].transform(X)
    clusterer = fitted_pipeline.named_steps["clusterer"]

    if hasattr(clusterer, "labels_"):
        labels = clusterer.labels_
    elif hasattr(clusterer, "predict"):
        labels = clusterer.predict(transformed)
    else:
        labels = clusterer.fit_predict(transformed)

    return transformed, np.asarray(labels)


def compute_metrics(transformed, labels):
    label_series = pd.Series(labels)
    valid_mask = label_series != -1
    transformed_eval = transformed[valid_mask.to_numpy()]
    labels_eval = label_series[valid_mask].to_numpy()

    if len(np.unique(labels_eval)) < 2:
        return None

    cluster_sizes = pd.Series(labels_eval).value_counts(normalize=True)
    noise_ratio = (label_series == -1).mean()

    return {
        "n_clusters_found": int(len(np.unique(labels_eval))),
        "noise_ratio": float((label_series == -1).mean()),
        "min_cluster_share": float(cluster_sizes.min()),
        "max_cluster_share": float(cluster_sizes.max()),
        "silhouette": float(silhouette_score(transformed_eval, labels_eval)),
        "calinski_harabasz": float(calinski_harabasz_score(transformed_eval, labels_eval)),
        "noise_ratio": float(noise_ratio),
    }

In [ ]:
search_results = []

for algorithm_name, spec in algorithm_spaces.items():
    for params in parameter_grid(spec["params"]):
        estimator = clone(spec["estimator"]).set_params(**params)
        pipeline = build_pipeline(estimator)
        pipeline.fit(search_data)

        transformed_search, labels_search = get_cluster_labels(pipeline, search_data)
        metrics = compute_metrics(transformed_search, labels_search)

        if metrics is None:
            continue

        search_results.append(
            {
                "algorithm": algorithm_name,
                "params": params,
                **metrics,
            }
        )

search_results_df = pd.DataFrame(search_results).sort_values(
    ["silhouette"],
    ascending=[False],
)

best_by_algorithm = (
    search_results_df.groupby("algorithm", as_index=False)
    .first()
    .sort_values("silhouette", ascending=False)
)
display(
    best_by_algorithm[
        [
            "algorithm",
            "params",
            "n_clusters_found",
            "min_cluster_share",
            "silhouette",
            "calinski_harabasz",
            "noise_ratio",
        ]
    ]
)